# Qwen3-8B AWQ INT4 test

이 노트북은 Qwen3-8B를 AWQ W4A16(INT4 weight, FP16 activation) 형식으로 양자화하고, 결과 `.pt` 파일을 Google Drive에 저장합니다.

권장 런타임: Colab GPU 런타임. Qwen3-8B 양자화는 T4 16GB에서는 메모리가 부족할 수 있고, A100/L4급 GPU가 더 안정적입니다.

## 1. Google Drive Mount

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Repository Setup

`REPO_URL`에는 Qwen3 수정사항이 반영된 본인 fork URL을 넣는 것을 권장합니다. 이미 `/content/llm-awq`에 코드를 업로드한 경우 `REPO_URL = ""` 그대로 두면 됩니다.

In [8]:
import os

REPO_URL = ""  # 예: "https://github.com/<your-id>/llm-awq.git"
REPO_DIR = "/content/drive/MyDrive/Global_capstone/jetson_test"

if REPO_URL and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if not os.path.exists(REPO_DIR):
    raise FileNotFoundError(
        f"{REPO_DIR}가 없습니다. REPO_URL에 fork URL을 넣거나, 수정된 llm-awq 폴더를 /content/llm-awq로 업로드하세요."
    )

%cd /content/drive/MyDrive/Global_capstone/jetson_test

/content/drive/MyDrive/Global_capstone/jetson_test


In [9]:
!python -m pip install -U pip
!python -m pip install -U pip setuptools wheel ninja
!python -m pip install --no-build-isolation --no-deps -e .
!python -m pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.2 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ninja]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
Obtaining file:///content/drive/MyDrive/Global_capstone/jetson_test
ERROR: file:///content/drive/MyDrive/Global_capstone/jetson_test does not appea

In [10]:
%cd /content/drive/MyDrive/Global_capstone/jetson_test/awq/kernels
!export TORCH_CUDA_ARCH_LIST="8.0"
!python setup.py install

/content/drive/MyDrive/Global_capstone/jetson_test/awq/kernels
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
running build
running build_ext
building 'awq_inference_engine' extension
ninja: no work to do.
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong

In [11]:
import os

extra = "/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/cuda/lib64"
old = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{extra}:{old}" if old else extra
print(os.environ["LD_LIBRARY_PATH"])

/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/cuda/lib64:/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/lib64-nvidia


In [12]:
%env LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/lib64-nvidia
!python -c "import awq_inference_engine; print('awq_inference_engine OK')"

env: LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/usr/lib64-nvidia
awq_inference_engine OK


In [13]:
!python3 -c "import torch; print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')"

True
NVIDIA A100-SXM4-40GB


In [14]:
TARGET_FILE_PATH = "/content/drive/MyDrive/Global_capstone/jetson_test/models/qwen3-4b-w4-g128-awq-v2.pt"
DESTINATION_DIR = "/content/"

!cp "{TARGET_FILE_PATH}" "{DESTINATION_DIR}"

print(f"File copied from {TARGET_FILE_PATH} to {DESTINATION_DIR}")

File copied from /content/drive/MyDrive/Global_capstone/jetson_test/models/qwen3-4b-w4-g128-awq-v2.pt to /content/


## using AWQ kernel

In [15]:
!python3 /content/drive/MyDrive/Global_capstone/jetson_test/infer_awq.py \
  --model_path /content/drive/MyDrive/Global_capstone/jetson_test/qwen3-4b-awq-runtime \
  --load_quant /content/qwen3-4b-w4-g128-awq-v2.pt \
  --prompt "please talk about harry potter" \
  --profile_fallback_ops

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
<think>
Okay, the user asked me to talk about Harry Potter. Let me start by recalling the main elements of the series. First, I should mention the author, J.K. Rowling, and the publication dates. The series is a fantasy novel series that's been very popular, right? It's about a young
awq_backend: kernel

[timing]
load_tokenizer: 1.6185s
load_config: 0.0037s
init_empty_model: 1.3939s
replace_linear_modules: 0.7564s
load_checkpoint_to_device: 2.7398s
model_to_eval: 0.0054s
build_inputs: 0.0227s
generate: 3.3274s
decode: 0.0003s
generated_tokens: 64
tokens_per_second: 19.23


# No kernel

In [17]:
!python3 /content/drive/MyDrive/Global_capstone/jetson_test/infer_awq.py \
  --model_path /content/drive/MyDrive/Global_capstone/jetson_test/qwen3-4b-awq-runtime \
  --load_quant /content/qwen3-4b-w4-g128-awq-v2.pt \
  --prompt "please talk about harry potter" \
  --profile_fallback_ops

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
<think>
Okay, the user asked me to talk about Harry Potter. Let me start by recalling the main elements of the series. First, I should mention the author, J.K. Rowling, and the publication dates. The series is a fantasy novel series that's been very popular, right? It's about a young
awq_backend: kernel

[timing]
load_tokenizer: 0.7167s
load_config: 0.0038s
init_empty_model: 0.4673s
replace_linear_modules: 0.0293s
load_checkpoint_to_device: 2.8637s
model_to_eval: 0.0052s
build_inputs: 0.0225s
generate: 2.9778s
decode: 0.0003s
generated_tokens: 64
tokens_per_second: 21.49
